# [LAB] LangGraph로 그래프 에이전트 만들기

받은 코드로 LangGraph 그래프를 다섯 단계로 만들어 봅니다. 빈칸(과제)은 AI와 함께 채우고, 완성된 코드(예제)는 실행하며 값을 바꿔 봅니다.

- **2장 기본 그래프** (과제): State, 노드 하나, START → chatbot → END를 잇고 컴파일합니다.
- **3장 조건부 엣지** (과제·예제): 질문의 언어를 감지해 한국어·영어 답변 노드로 나눠 보냅니다.
- **4장 ToolNode로 ReAct 루프** (과제·예제): 도구 호출 → 도구 실행 → 다시 판단하는 반복을 그래프로 만듭니다.
- **5장 사람 승인** (과제): 도구 실행 직전에 멈추고, 승인하면 이어 가고 거부하면 막습니다.
- **6장 대화형 승인** (예제): 사람이 입력창에서 직접 승인·거부를 고릅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**목차**

- 1. 실습 방법
- 2. 기본 그래프 만들기 (과제)
  - 2.1 State와 chatbot 노드 만들기 (과제)
  - 2.2 노드와 엣지 잇기 (과제)
  - 2.3 컴파일하고 실행하기 (과제)
- 3. 조건부 엣지로 언어별 분기하기 (예제) (과제)
  - 3.1 State와 노드 정의 (과제)
  - 3.2 라우팅 함수 (예제)
  - 3.3 그래프 조립 (과제)
  - 3.4 질문 넣어 실행하기 (예제)
- 4. ToolNode로 ReAct 루프 만들기 (예제) (과제)
  - 4.1 도구 정의 (예제)
  - 4.2 도구 목록과 LLM 연결 (과제)
  - 4.3 agent 노드 (과제)
  - 4.4 ToolNode로 루프 잇고 실행하기 (과제)
  - 4.5 create_agent로 한 번에 만들기 (예제)
- 5. 사람 승인 넣기 (예제) (과제)
  - 5.1 도구와 그래프 조립 (예제)
  - 5.2 체크포인터와 중단 지점 (과제)
  - 5.3 1단계: 도구 실행 직전에 멈추기 (예제)
  - 5.4 2단계: 승인하고 이어 가기 (예제)
  - 5.5 거부하고 이어 가기 (예제)
- 6. 대화형 승인 에이전트 (예제)
  - 6.1 도구와 그래프 조립 (예제)
  - 6.2 승인·거부를 묻는 함수 (예제)
  - 6.3 에이전트 실행 루프 (예제)
  - 6.4 직접 실행하기 (예제)
  - 6.5 AI에게 물어보기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

## 2. 기본 그래프 만들기 <small>(과제)</small>

노드 하나짜리 가장 작은 그래프를 만듭니다. State(공유 데이터)와 Node(작업 단위)를 정하고, Edge(다음 이동)로 START → chatbot → END를 이은 뒤, 컴파일해서 질문 하나를 실행합니다.

빈칸 세 곳은 모두 과제입니다. 위에서부터 한 셀씩 채우고 실행합니다.

준비 코드입니다. 그래프를 만드는 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접어 두었으니 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 2.1 State와 chatbot 노드 만들기 <small>(과제)</small>

**✍️ WRITE_HERE: State의 messages를 LLM에 전달하고 응답을 messages에 담아 반환해 주세요.**

노드는 State를 받아서, 바꿀 값만 담은 딕셔너리를 돌려주는 함수입니다. `chatbot_node`는 지금까지의 메시지 목록을 LLM에 넘기고, LLM의 답 메시지를 `messages`에 담아 돌려줘야 합니다. `add_messages` 규칙 덕분에 돌려준 메시지는 기존 목록 뒤에 쌓입니다.

**프롬프트 힌트** 빈칸에 State의 메시지 목록을 `llm`에 넘기고, 받은 응답을 `messages` 키의 목록에 담아 돌려주는 코드를 써 달라고 합니다. 빈칸 밖의 줄은 고치지 말라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(함수만 정의하므로 출력은 없습니다)
- 🟢 2.3을 실행하면 마지막 줄에 LangGraph를 설명하는 답변이 출력된다
- ⛔ 2.3의 마지막 줄에 답변 대신 질문이 그대로 다시 출력된다 → 노드가 LLM의 응답을 돌려주는지 확인합니다
- ⛔ `SyntaxError`나 `IndentationError`가 난다 → 채운 줄이 함수 안쪽(4칸 들여쓰기)에 있는지 확인합니다

In [ ]:
# ── LangGraph 3대 요소 ──────────────────────────────────────────────────────

# 1. State: 그래프 전체에서 공유되는 데이터 구조.
#    TypedDict 또는 Pydantic BaseModel로 정의한다.
#    add_messages Reducer 덕분에 messages는 덮어쓰기가 아닌 누적(append)이다.


class State(TypedDict):
    messages: Annotated[list, add_messages]


# 2. Node: State를 받아 State의 일부(dict)를 반환하는 함수.
#    반환된 dict는 Reducer 규칙에 따라 기존 State에 병합된다.
def chatbot_node(state: State) -> dict:
    # ✍️ WRITE_HERE: State의 messages를 LLM에 전달하고 응답을 messages에 담아 반환해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
response = llm.invoke(state["messages"])
return {"messages": [response]}
```

</details>

### 2.2 노드와 엣지 잇기 <small>(과제)</small>

**✍️ WRITE_HERE: chatbot 노드를 추가하고 START에서 chatbot, chatbot에서 END로 이어 주세요.**

`builder`에 노드를 올리고 엣지로 잇습니다. `add_node`에는 노드 이름과 그 노드가 할 일(함수)을 주고, `add_edge`에는 출발 노드와 도착 노드를 줍니다. `START`와 `END`는 LangGraph가 주는 시작점과 끝점입니다.

**프롬프트 힌트** 빈칸에 `chatbot_node` 함수를 `"chatbot"`이라는 이름의 노드로 추가하고, START → chatbot → END 순서로 엣지를 잇는 코드를 써 달라고 합니다. 노드 이름은 `"chatbot"` 그대로 쓰라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(노드와 엣지만 더하므로 출력은 없습니다)
- 🟢 2.3의 그래프 구조 출력에 `__start__ --> chatbot;`과 `chatbot --> __end__;`가 보인다
- ⛔ 2.3에서 `Graph must have an entrypoint` 오류가 난다 → START에서 chatbot으로 가는 엣지가 있는지 확인합니다

In [ ]:
# 3. Edge: 노드 간 흐름을 정의한다.
#    START/END는 LangGraph가 제공하는 진입·종료 지점이다.
builder = StateGraph(State)
# ✍️ WRITE_HERE: chatbot 노드를 추가하고 START에서 chatbot, chatbot에서 END로 이어 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
```

</details>

### 2.3 컴파일하고 실행하기 <small>(과제)</small>

**✍️ WRITE_HERE: 그래프를 컴파일해 실행 가능한 app을 만들어 주세요.**

`compile()`은 조립한 그래프를 실행할 수 있는 `app`으로 만듭니다. 이 셀은 컴파일한 뒤 그래프 구조를 글자(Mermaid 형식)로 출력하고, 질문 하나를 넣어 실행합니다.

**프롬프트 힌트** 빈칸에 `builder`를 컴파일해 `app`이라는 이름에 담는 한 줄을 써 달라고 합니다. 아래 실행 코드는 그대로 두라고 적습니다.

**성공 기준**

- 🟢 그래프 구조 출력에 `__start__ --> chatbot;`과 `chatbot --> __end__;`가 보인다
- 🟢 마지막 줄에 LangGraph 설명이 두 문장 안팎으로 출력된다
- ⛔ `NameError: name 'app' is not defined`가 난다 → 컴파일 결과를 `app`에 담았는지 확인합니다

In [ ]:
# ✍️ WRITE_HERE: 그래프를 컴파일해 실행 가능한 app을 만들어 주세요.
...  # ✍️ 여기를 채웁니다


# 그래프 구조 출력 (Mermaid 다이어그램)
print(app.get_graph().draw_mermaid())
print()

result = app.invoke(
    {"messages": [HumanMessage(content="LangGraph가 뭔지 두 문장으로 설명해줘.")]}
)
print(result["messages"][-1].content)

<details><summary>정답 보기</summary>

```python
app = builder.compile()
```

</details>

**볼 지점**
- 그래프 구조 출력의 `__start__`와 `__end__`는 LangGraph가 붙이는 시작점과 끝점입니다. 그 사이에 추가한 `chatbot` 노드가 있습니다.
- 마지막 줄이 LLM의 답변 텍스트입니다. `result["messages"]`에는 질문과 답이 차례로 쌓여 있고, 코드는 그중 마지막(`[-1]`)을 꺼냈습니다.

## 3. 조건부 엣지로 언어별 분기하기 <small>(예제) (과제)</small>

질문의 언어를 먼저 알아낸 뒤, 한국어면 한국어 답변 노드로, 아니면 영어 답변 노드로 보냅니다. 갈 곳을 함수의 결과로 정하는 조건부 엣지를 씁니다.

빈칸 두 곳은 과제입니다. 마지막 실행 셀은 예제라서, 채운 뒤 `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 다시 실행해 봅니다.

이 장의 준비 코드입니다. 앞 장과 같은 도구에 `Literal`(정해진 값만 허용하는 타입 표시)을 더해 불러옵니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 3.1 State와 노드 정의 <small>(과제)</small>

**✍️ WRITE_HERE: LLM으로 언어를 감지하고 'ko' 또는 'en' 값을 정해 주세요.**

`detect_language` 노드는 마지막 메시지의 언어를 LLM에게 물어 `"ko"`나 `"en"`으로 정하고, State의 `language` 필드에 기록합니다. 답 메시지가 아니라 `language` 값만 돌려주는 노드입니다. LLM의 답에는 다른 글자가 섞일 수 있으니, 답을 `"ko"`와 `"en"` 둘 중 하나로 정리해야 다음 라우팅 함수가 제대로 고릅니다.

**프롬프트 힌트** 빈칸에 `last`의 언어를 "ko"나 "en" 중 하나로만 답하라고 `llm`에 묻고, 답을 소문자로 바꿔 `ko`가 들어 있으면 `"ko"`, 아니면 `"en"`을 `lang`에 담는 코드를 써 달라고 합니다. 바로 아래 `return` 줄은 그대로 두라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(노드 함수만 정의합니다)
- 🟢 3.4에서 한국어 질문은 `감지 언어: ko`, 영어 질문은 `감지 언어: en`으로 출력된다
- ⛔ 3.4에서 두 질문이 모두 같은 언어로 감지된다 → LLM의 답에서 `ko`를 찾는 줄을 확인합니다
- ⛔ 3.4에서 `NameError: name 'lang' is not defined`가 난다 → 판단한 값을 `lang`에 담았는지 확인합니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    language: str  # 감지된 언어 ('ko' | 'en')


# ── 노드 정의 ────────────────────────────────────────────────────────────────


def detect_language(state: State) -> dict:
    """마지막 메시지의 언어를 감지해 State에 기록한다."""
    last = state["messages"][-1].content
    # ✍️ WRITE_HERE: LLM으로 언어를 감지하고 'ko' 또는 'en' 값을 정해 주세요.
    ...  # ✍️ 여기를 채웁니다
    return {"language": lang}


def korean_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "반드시 한국어로만 답변하세요."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


def english_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "Respond only in English."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}

<details><summary>정답 보기</summary>

```python
result = llm.invoke(
    f'다음 텍스트의 언어를 "ko" 또는 "en" 중 하나로만 판별해: {last}'
)
lang = "ko" if "ko" in result.content.lower() else "en"
```

</details>

### 3.2 라우팅 함수 <small>(예제)</small>

In [ ]:
# ── 라우팅 함수 ──────────────────────────────────────────────────────────────
# add_conditional_edges의 두 번째 인자로 전달되는 함수.
# State를 받아 다음에 실행할 노드 이름(또는 END)을 반환한다.
def route_by_language(state: State) -> Literal["korean_response", "english_response"]:
    return "korean_response" if state.get("language") == "ko" else "english_response"

### 3.3 그래프 조립 <small>(과제)</small>

**✍️ WRITE_HERE: detect_language 이후 조건부 엣지를 추가해 주세요.**

보통 엣지(`add_edge`)는 도착 노드가 정해져 있습니다. 조건부 엣지(`add_conditional_edges`)는 출발 노드가 끝난 뒤 라우팅 함수를 실행하고, 그 함수가 돌려준 이름의 노드로 갑니다. 3.2의 `route_by_language`가 `"korean_response"`나 `"english_response"`를 돌려줍니다.

**프롬프트 힌트** 빈칸에 `detect_language` 노드가 끝나면 `route_by_language` 함수의 결과에 따라 다음 노드로 가는 조건부 엣지를 추가하는 한 줄을 써 달라고 합니다. 두 응답 노드로 동시에 보내는 방식은 쓰지 말라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(그래프를 조립만 합니다)
- 🟢 3.4에서 질문마다 답(`A:`)이 하나씩, 질문과 같은 언어로 출력된다
- ⛔ 3.4에서 `A:` 뒤에 답 대신 질문이 그대로 출력된다 → detect_language 뒤에 이어지는 엣지가 없는 것입니다. 조건부 엣지의 출발 노드 이름을 확인합니다

In [ ]:
# ── 그래프 조립 ──────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("detect_language", detect_language)
builder.add_node("korean_response", korean_response)
builder.add_node("english_response", english_response)

builder.add_edge(START, "detect_language")

# 조건부 엣지: detect_language 노드 이후 route_by_language 결과에 따라 분기
# ✍️ WRITE_HERE: detect_language 이후 조건부 엣지를 추가해 주세요.
...  # ✍️ 여기를 채웁니다

builder.add_edge("korean_response", END)
builder.add_edge("english_response", END)

app = builder.compile()

<details><summary>정답 보기</summary>

```python
builder.add_conditional_edges("detect_language", route_by_language)
```

</details>

### 3.4 질문 넣어 실행하기 <small>(예제)</small>

**✍️ CHANGE_HERE: 두 번째 질문을 다른 언어로 바꿔 봅니다**

먼저 그대로 실행해 두 질문의 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 영어 질문을 일본어 같은 다른 언어의 질문으로 바꿔 다시 실행합니다. 따옴표와 쉼표는 지우지 않습니다.

In [ ]:
# ── 실행 ─────────────────────────────────────────────────────────────────────
for question in [
    "안녕하세요! LangGraph에 대해 알려주세요.",
    # ✍️ CHANGE_HERE: 두 번째 질문을 다른 언어로 바꿔 봅니다
    "Hello! Tell me about LangGraph.",
]:
    result = app.invoke({"messages": [HumanMessage(content=question)], "language": ""})
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content}")
    print(f"감지 언어: {result['language']}")
    print()

**성공 기준**

- 🟢 질문마다 `Q:`, `A:`, `감지 언어:`가 한 번씩 출력된다
- 🟢 바꾼 질문에도 `감지 언어:`는 `ko`나 `en` 중 하나로만 출력되고, 답도 그 언어로 나온다(이 그래프의 갈래는 두 개뿐입니다)

**볼 지점**
- `감지 언어:` 줄이 라우팅 함수가 본 값입니다. `ko`면 한국어 답변 노드, 아니면 영어 답변 노드로 갔습니다.
- 두 답변 노드는 system 메시지만 다릅니다. 같은 메시지 목록 앞에 "한국어로만" 또는 "English only" 지시를 붙여 LLM을 호출합니다.

## 4. ToolNode로 ReAct 루프 만들기 <small>(예제) (과제)</small>

도구를 가진 에이전트를 그래프로 만듭니다. agent 노드가 도구를 쓸지 판단하고, ToolNode가 도구를 실행하고, 다시 agent로 돌아가 결과를 보고 답을 이어 씁니다. 판단 → 도구 실행 → 관찰을 되풀이하는 ReAct 루프입니다. 도구 실행 뒤 LLM을 다시 호출하는 반복을 직접 짜지 않고 `ToolNode`와 `tools_condition`에 맡깁니다.

빈칸 세 곳은 과제입니다. 도구 정의 셀과 마지막 `create_agent` 셀은 예제입니다.

준비 코드입니다. 그래프 도구와 `ToolNode`·`tools_condition`, 에이전트를 한 번에 만드는 `create_agent`를 불러옵니다. 모델은 4.2에서 도구를 연결하며 준비합니다. 실행만 하면 됩니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

### 4.1 도구 정의 <small>(예제)</small>

**✍️ CHANGE_HERE: 달러→원 환율을 바꿔 봅니다**

먼저 그대로 두고 이 장을 4.4까지 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `1350.0`을 다른 값(예: `1500.0`)으로 바꾸고, 이 셀부터 4.4까지 차례로 다시 실행합니다.

In [ ]:
@tool
def get_exchange_rate(base_currency: str, target_currency: str) -> float:
    """고정 환율을 반환한다. base_currency와 target_currency는 USD, KRW, EUR 중 하나."""
    rates = {
        # ✍️ CHANGE_HERE: 달러→원 환율을 바꿔 봅니다
        ("USD", "KRW"): 1350.0,
        ("KRW", "USD"): 1 / 1350.0,
        ("EUR", "KRW"): 1470.0,
        ("KRW", "EUR"): 1 / 1470.0,
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
    }
    if base_currency == target_currency:
        return 1.0
    return rates.get((base_currency, target_currency), -1.0)


@tool
def calculate(expression: str) -> float:
    """간단한 수식을 계산한다. 예: '1350 * 100'"""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        raise ValueError("허용되지 않는 문자가 포함되어 있습니다.")
    return eval(expression)  # noqa: S307

**성공 기준**

- 🟢 환율을 바꾸고 4.4까지 다시 실행하면 최종 답의 금액도 바꾼 환율로 계산한 값으로 바뀐다(모델이 도구 결과를 보고 답했다는 뜻입니다)

### 4.2 도구 목록과 LLM 연결 <small>(과제)</small>

**✍️ WRITE_HERE: 도구 목록을 만들고 bind_tools()로 LLM에 연결해 주세요.**

LLM이 도구를 쓰려면 어떤 도구가 있는지 알아야 합니다. `bind_tools()`는 도구의 이름·설명·인자 형식을 LLM에 알려 줍니다. 같은 도구 목록 `tools`를 4.4의 ToolNode도 씁니다.

**프롬프트 힌트** 빈칸에 위에서 정의한 두 도구를 담은 목록 `tools`를 만들고, `get_llm()`으로 받은 모델에 `bind_tools()`로 이 목록을 연결해 `llm`에 담는 코드를 써 달라고 합니다. 이름 `tools`와 `llm`은 아래 셀에서 쓰므로 바꾸지 말라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(도구를 연결만 합니다)
- 🟢 4.4 실행 결과의 `[AIMessage]` 줄에 `get_exchange_rate` 호출이 보인다
- ⛔ 4.4에서 `[ToolMessage]` 줄 없이 바로 최종 답이 나온다 → 도구를 LLM에 연결했는지 확인합니다

In [ ]:
# ✍️ WRITE_HERE: 도구 목록을 만들고 bind_tools()로 LLM에 연결해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
tools = [get_exchange_rate, calculate]
llm = get_llm().bind_tools(tools)
```

</details>

### 4.3 agent 노드 <small>(과제)</small>

**✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

`agent_node`는 지금까지의 메시지를 도구가 연결된 `llm`에 넘기고, 받은 AI 메시지를 `messages`에 담아 돌려줍니다. AI 메시지에 도구 호출(`tool_calls`)이 있으면 다음에 ToolNode가 실행되고, 없으면 최종 답으로 끝납니다.

**프롬프트 힌트** 빈칸에 State의 메시지 목록을 도구가 연결된 `llm`에 넘기고, 응답을 `messages` 키의 목록에 담아 돌려주는 한 줄을 써 달라고 합니다. 함수 이름과 들여쓰기는 그대로 두라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(agent 노드 함수만 정의합니다)
- 🟢 4.4 실행 결과에 `[HumanMessage]`, `[AIMessage]`, `[ToolMessage]`가 차례로 보인다
- ⛔ 4.4에서 `[HumanMessage]` 한 줄만 출력되고 끝난다 → 노드가 LLM의 응답을 돌려주는지 확인합니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    # ✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
return {"messages": [llm.invoke(state["messages"])]}
```

</details>

### 4.4 ToolNode로 루프 잇고 실행하기 <small>(과제)</small>

**✍️ WRITE_HERE: ToolNode로 tools 노드를 추가하고, 도구 실행 뒤 agent로 돌아가는 엣지를 이어 주세요.**

ToolNode는 AI 메시지의 도구 호출을 읽어 해당 도구를 실행하고, 결과를 ToolMessage(도구 결과 메시지)로 돌려주는 미리 만들어진 노드입니다. `tools_condition`은 마지막 메시지에 도구 호출이 있으면 `"tools"` 노드로, 없으면 끝으로 보냅니다. 도구를 실행한 뒤 agent로 돌아가는 엣지가 있어야 모델이 도구 결과를 보고 답을 이어 씁니다. 이 되돌아가는 엣지가 순환 그래프의 고리입니다.

**프롬프트 힌트** 빈칸에 `tools` 목록으로 ToolNode를 만들어 `"tools"`라는 이름의 노드로 추가하고, `"tools"`에서 `"agent"`로 돌아가는 엣지를 잇는 코드를 써 달라고 합니다. 노드 이름은 `tools_condition`이 찾는 `"tools"` 그대로 쓰라고 적습니다.

**성공 기준**

- 🟢 메시지 목록이 `[HumanMessage]` → `[AIMessage]`(도구 호출) → `[ToolMessage]` → … → `[AIMessage]`(최종 답) 순서로 출력된다
- 🟢 최종 답의 금액이 도구의 환율로 계산한 값(100 × 1350 = 135,000원)과 맞는다
- ⛔ `[ToolMessage]` 뒤에서 끝나고 최종 답이 없다 → tools에서 agent로 돌아가는 엣지를 확인합니다
- ⛔ `tools`라는 노드를 찾을 수 없다는 오류가 난다 → 노드 이름이 `"tools"`인지 확인합니다

In [ ]:
# ToolNode: AI 메시지에 포함된 tool_calls를 파싱하고 해당 도구를 실행한 뒤
#           결과를 ToolMessage 리스트로 반환한다.
# tools_condition: 마지막 메시지에 tool_calls가 있으면 'tools'로,
#                  없으면(최종 답변이면) END로 라우팅한다.
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # → 'tools' or END
# ✍️ WRITE_HERE: ToolNode로 tools 노드를 추가하고, 도구 실행 뒤 agent로 돌아가는 엣지를 이어 주세요.
...  # ✍️ 여기를 채웁니다

app = builder.compile()

print("=== 직접 조립한 ReAct Agent ===")
result = app.invoke(
    {"messages": [HumanMessage(content="100달러를 원화로 환산하면 얼마야?")]}
)
for msg in result["messages"]:
    role = msg.__class__.__name__
    content = msg.content or str(getattr(msg, "tool_calls", ""))
    print(f"  [{role}] {content}")

print()

<details><summary>정답 보기</summary>

```python
tool_node = ToolNode(tools)
builder.add_node("tools", tool_node)
builder.add_edge("tools", "agent")
```

</details>

### 4.5 create_agent로 한 번에 만들기 <small>(예제)</small>

In [ ]:
# ── create_agent: 위와 동일한 ReAct 루프를 한 줄로 만드는 편의 함수 ────────────
# 커스텀 노드·엣지가 필요 없을 때 사용. 내부적으로 위와 같은 그래프를 자동 생성한다.
# 멀티에이전트처럼 여러 에이전트가 서로 통신하는 구조는 StateGraph 직접 조립이 필요하다.
print("=== create_agent ===")
agent = create_agent(get_llm(), tools)
result2 = agent.invoke({"messages": [HumanMessage(content="50유로는 몇 원이야?")]})
print(result2["messages"][-1].content)

**볼 지점**
- 4.4의 출력에서 `[AIMessage]` 줄이 두 번 이상 나옵니다. 앞의 것은 도구 호출이고, 마지막 것이 최종 답입니다. 도구 결과가 `[ToolMessage]`로 들어온 뒤 agent가 다시 실행된 것입니다.
- 4.5의 `create_agent`는 4.4에서 직접 조립한 agent 노드, tools 노드, 돌아가는 엣지를 한 번에 만들어 줍니다. 노드를 더하거나 흐름을 바꿔야 할 때는 4.4처럼 직접 조립합니다.

## 5. 사람 승인 넣기 <small>(예제) (과제)</small>

메일 보내기 도구를 가진 에이전트가 도구를 실행하기 직전에 멈추게 합니다. 사람이 대기 중인 도구 호출을 확인한 뒤 승인하면 이어서 실행하고, 거부하면 거부 결과를 State에 넣고 이어 갑니다.

`send_email` 도구는 `[실제 전송]` 줄을 화면에 출력만 하고 실제 메일은 보내지 않습니다. 이 줄이 보이면 "도구가 실행되었다"는 뜻입니다.

5.2의 빈칸 두 곳은 과제입니다. 5.3 → 5.4 → 5.5는 같은 대화를 이어 가므로 차례로 실행합니다. 처음부터 다시 하려면 5.2부터 실행합니다.

이 장의 준비 코드입니다. 체크포인터(`MemorySaver`)와 도구 결과 메시지(`ToolMessage`)를 함께 불러오고 모델을 준비합니다. 실행을 잊지 않습니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 5.1 도구와 그래프 조립 <small>(예제)</small>

In [ ]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송한다."""
    print(f"\n  [실제 전송] {recipient} / 제목: {subject}")
    return f"이메일이 {recipient}에게 성공적으로 전송되었습니다."


tools = [send_email]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent");

### 5.2 체크포인터와 중단 지점 <small>(과제)</small>

**✍️ WRITE_HERE: MemorySaver 체크포인터를 생성해 주세요.**

Checkpointer는 실행 상태를 저장합니다. 이 실습에서는 메모리에 저장하는 `MemorySaver`를 씁니다. 저장된 상태가 있어야 멈춘 지점부터 이어 갈 수 있습니다.

**프롬프트 힌트** 빈칸에 메모리에 상태를 저장하는 `MemorySaver` 체크포인터를 만들어 `checkpointer`에 담는 한 줄을 써 달라고 합니다. 이미 불러온 도구만 쓰라고 적습니다.

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(체크포인터를 만들고 컴파일만 하므로 출력은 없습니다)
- 🟢 5.3 실행 뒤 "대기 중인 도구 호출" 아래에 `send_email(...)`이 보인다
- ⛔ 5.3에서 `No checkpointer set` 오류가 난다 → `checkpointer`를 만들어 컴파일에 넘겼는지 확인합니다

**✍️ WRITE_HERE: tools 노드 실행 직전에 중단되도록 그래프를 컴파일해 주세요.**

`interrupt_before`에 노드 이름을 주면 그 노드를 실행하기 직전에 그래프가 멈춥니다. 메일을 실제로 보내는 `tools` 노드 앞에서 멈춰야 사람이 확인할 수 있습니다. 컴파일할 때 바로 위의 `checkpointer`도 함께 넘깁니다.

**프롬프트 힌트** 빈칸에 `builder`를 컴파일하되, 위에서 만든 `checkpointer`를 쓰고 `"tools"` 노드 실행 직전에 멈추도록 설정해 `app`에 담는 한 줄을 써 달라고 합니다. 새 노드나 함수는 만들지 말라고 적습니다.

**성공 기준**

- 🟢 5.3 실행 뒤에는 `[실제 전송]` 줄이 없고, 대기 중인 도구 호출만 보인다
- 🟢 5.4를 실행한 뒤에야 `[실제 전송] kim@example.com` 줄이 한 번 출력된다
- ⛔ 5.3에서 이미 `[실제 전송]`이 출력된다 → 멈출 노드 이름이 `"tools"`인지 확인합니다
- ⛔ "대기 중인 도구 호출" 아래가 비어 있다 → 모델이 도구를 호출하지 않은 것입니다. 5.2부터 다시 실행합니다

In [ ]:
# ✍️ WRITE_HERE: MemorySaver 체크포인터를 생성해 주세요.
...  # ✍️ 여기를 채웁니다

# interrupt_before: 지정한 노드 실행 직전에 그래프를 일시 중단한다.
# 사용자가 도구 호출 내용을 확인하고 승인/거부할 수 있다.
# ✍️ WRITE_HERE: tools 노드 실행 직전에 중단되도록 그래프를 컴파일해 주세요.
...  # ✍️ 여기를 채웁니다

config = {"configurable": {"thread_id": "hitl_demo"}}

<details><summary>정답 보기</summary>

```python
checkpointer = MemorySaver()
```

```python
app = builder.compile(checkpointer=checkpointer, interrupt_before=["tools"])
```

</details>

### 5.3 1단계: 도구 실행 직전에 멈추기 <small>(예제)</small>

In [ ]:
# ── 1단계: tools 노드 직전까지 실행 ──────────────────────────────────────────
print("=== 1단계: 에이전트 실행 (tools 직전에서 중단) ===")
app.invoke(
    {
        "messages": [
            HumanMessage(
                content='kim@example.com에게 "오늘 회의 취소" 제목으로 "오후 3시 회의가 취소되었습니다"라는 이메일을 보내줘.'
            )
        ]
    },
    config=config,
)

state = app.get_state(config)
pending_tools = state.values["messages"][-1].tool_calls
print(f"\n대기 중인 도구 호출:")
for tc in pending_tools:
    print(f"  - {tc['name']}({tc['args']})")

### 5.4 2단계: 승인하고 이어 가기 <small>(예제)</small>

In [ ]:
# ── 2단계: 사용자 승인 후 재개 ────────────────────────────────────────────────
print("\n=== 2단계: 승인 후 재개 ===")
# None을 입력하면 마지막으로 중단된 지점부터 그래프가 이어서 실행된다.
result = app.invoke(None, config=config)
print(result["messages"][-1].content)

### 5.5 거부하고 이어 가기 <small>(예제)</small>

In [ ]:
# ── 거부 시나리오 ─────────────────────────────────────────────────────────────
print("\n=== 거부 시나리오 ===")
config2 = {"configurable": {"thread_id": "hitl_reject"}}
app.invoke(
    {
        "messages": [
            HumanMessage(
                content='boss@example.com에게 "긴급" 제목으로 "즉시 연락 바랍니다"라는 이메일을 보내줘.'
            )
        ]
    },
    config=config2,
)

# 거부: tool_calls를 직접 ToolMessage로 처리해 상태에 삽입한 뒤 재개
state2 = app.get_state(config2)
rejected_calls = state2.values["messages"][-1].tool_calls

rejection_messages = [
    ToolMessage(content="사용자가 이메일 전송을 취소했습니다.", tool_call_id=tc["id"])
    for tc in rejected_calls
]
app.update_state(config2, {"messages": rejection_messages}, as_node="tools")
result2 = app.invoke(None, config=config2)
print(result2["messages"][-1].content)

**성공 기준**

- 🟢 `=== 거부 시나리오 ===` 아래에 `[실제 전송] boss@example.com` 줄이 없다
- 🟢 마지막 답이 메일을 보내지 않았다(취소되었다)는 내용이다

**볼 지점**
- 승인 흐름에서는 1단계 뒤에 멈췄다가 `app.invoke(None, ...)`로 멈춘 곳부터 이어 갔습니다.
- 거부 흐름에서는 `app.update_state(...)`가 거부 메시지를 도구 결과처럼 State에 넣은 뒤 이어 갔습니다. 그래서 `send_email`이 실행되지 않았습니다.

---

**AI에게 물어보기** 5.5의 `app.update_state` 줄이 무엇을 막는지 AI에게 리뷰를 받습니다.

**프롬프트 힌트** 5.5 셀을 붙여 넣고, `app.update_state` 줄이 없으면 이어서 실행할 때 무슨 일이 생기는지, 출력의 어느 줄로 알 수 있는지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**
- 🟢 그 줄이 없으면 멈춘 지점부터 그대로 이어져 `send_email`이 실행되고 `[실제 전송] boss@example.com`이 출력된다고 짚었다
- 🟢 거부 메시지를 만들기만 해서는 State에 들어가지 않는다고 설명했다
- ⛔ "거부 메시지를 만들었으니 보내지 않는다"고 답했다 → `rejection_messages`가 어디에 쓰이는지 다시 묻습니다

## 6. 대화형 승인 에이전트 <small>(예제)</small>

5장의 승인·거부를 사람이 직접 고르는 프로그램으로 만듭니다. 에이전트가 도구를 쓰려고 할 때마다 멈추고, 입력창에서 `y`(승인), `n`(거부), `q`(종료) 중 하나를 고릅니다.

코드는 완성되어 있는 예제입니다. 6.1~6.3을 실행해 그래프와 함수를 준비하고, 6.4를 실행해 직접 요청해 봅니다. 6.4를 실행하면 입력창이 열립니다. 요청(예: 메일 보내기)을 적고, 이어서 `y`/`n`/`q`를 입력합니다. `n`을 고르면 거부 사유도 묻습니다. 끝나지 않으면 `q`를 입력하거나 실행을 중지(■)합니다.

준비 코드는 5장과 같은 도구를 불러오고 모델을 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 6.1 도구와 그래프 조립 <small>(예제)</small>

In [ ]:
@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """이메일을 전송한다."""
    print(f"\n  [실제 전송] {recipient} / 제목: {subject}")
    return f"이메일이 {recipient}에게 성공적으로 전송되었습니다."


tools = [send_email]
llm_with_tools = llm.bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

app = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],
)

### 6.2 승인·거부를 묻는 함수 <small>(예제)</small>

In [ ]:
def print_tool_calls(tool_calls: list[dict]) -> None:
    print("\n실행 대기 중인 도구 호출")
    for index, tool_call in enumerate(tool_calls, start=1):
        print(f"{index}. {tool_call['name']}")
        for key, value in tool_call["args"].items():
            print(f"   - {key}: {value}")


def reject_tool_calls(config: dict, tool_calls: list[dict], reason: str) -> None:
    rejection_messages = [
        ToolMessage(
            content=f"사용자가 도구 실행을 거부했습니다. 사유: {reason}",
            tool_call_id=tool_call["id"],
        )
        for tool_call in tool_calls
    ]
    app.update_state(config, {"messages": rejection_messages}, as_node="tools")


def ask_human_decision() -> tuple[str, str]:
    print("\n도구를 실행할까요?")
    print("- y: 승인 후 실행")
    print("- n: 거부 후 에이전트에게 취소 사유 전달")
    print("- q: 프로그램 종료")

    decision = input("선택(y/n/q): ").strip().lower()
    if decision == "n":
        reason = input("거부 사유를 입력하세요: ").strip()
        return decision, reason
    return decision, ""

### 6.3 에이전트 실행 루프 <small>(예제)</small>

In [ ]:
def run_agent(user_input: str, thread_id: str = "hitl_demo") -> None:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)

    while True:
        state = app.get_state(config)
        tool_calls = state.values["messages"][-1].tool_calls

        if not tool_calls:
            print("\n에이전트 응답")
            print(result["messages"][-1].content)
            break

        print_tool_calls(tool_calls)
        decision, reason = ask_human_decision()

        if decision == "y":
            result = app.invoke(None, config=config)
        elif decision == "n":
            reject_tool_calls(config, tool_calls, reason or "사유 없음")
            result = app.invoke(None, config=config)
        elif decision == "q":
            print("프로그램을 종료합니다.")
            break
        else:
            print("y, n, q 중 하나를 입력하세요.")

### 6.4 직접 실행하기 <small>(예제)</small>

In [ ]:
if __name__ == "__main__":
    request = input("에이전트에게 요청할 내용을 입력하세요: ")
    run_agent(request)

**성공 기준**

- 🟢 요청을 적으면 "실행 대기 중인 도구 호출"과 받는 사람·제목·본문이 출력된다
- 🟢 `y`를 고르면 `[실제 전송]`이 출력되고, `n`을 고르면 출력되지 않고 거부 사유를 반영한 답이 나온다

**볼 지점**
- `run_agent`는 멈출 때마다 State의 마지막 메시지에서 도구 호출을 꺼내 보여 줍니다. 사람이 고른 값에 따라 이어 가거나(`y`), 거부 메시지를 넣고 이어 가거나(`n`), 끝냅니다(`q`).
- `n`을 고르면 `reject_tool_calls`가 5장의 거부와 같은 방식(`app.update_state`)으로 거부 사유를 State에 기록합니다.

### 6.5 AI에게 물어보기

**프롬프트 힌트** 6.2와 6.3 셀을 붙여 넣고, `n`을 고르면 어떤 메시지가 State에 들어가며 재개 뒤 에이전트가 왜 메일을 보내지 않는지 코드 줄을 짚어 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 AI 설명이 `app.update_state(..., as_node="tools")` 줄을 짚었다
- 🟢 거부 사유가 담긴 ToolMessage가 도구 결과 자리에 들어간다고 설명했다
- ⛔ 6.4에서 `n`을 골랐는데 `[실제 전송]`이 출력되었다 → 입력한 값이 `n`인지 확인하고 6.4를 다시 실행합니다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 7. 정리

받은 코드로 LangGraph 그래프를 다섯 단계로 만들었습니다.

| 장 | 만든 것 | 핵심 코드 |
|---|---|---|
| 2. 기본 그래프 | START → chatbot → END | `add_node`, `add_edge`, `compile` |
| 3. 조건부 엣지 | 언어에 따라 한국어·영어 노드로 분기 | `add_conditional_edges` |
| 4. ToolNode | 도구 호출 → 도구 실행 → 다시 판단하는 루프 | `ToolNode`, `tools_condition` |
| 5. 사람 승인 | 도구 실행 직전에 멈추고 승인·거부 | `MemorySaver`, `interrupt_before`, `update_state` |
| 6. 대화형 승인 | 사람이 y/n/q로 고르는 실행 루프 | `get_state`, `input()` |

그래프는 노드를 더하고 엣지로 이어 조립합니다. 반복은 되돌아가는 엣지로, 분기는 조건부 엣지로, 사람 승인은 Checkpointer와 중단 지점으로 만듭니다.

빈칸은 AI와 함께 채우고, 실행 결과를 성공 기준(🟢/⛔)과 비교한 뒤 정답 보기로 확인했습니다.